# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. The schema describes mass spectrometry analysis results from extracellular vesicles in human mast cells, annotated with standard metadata, fields, record sets, and measurements.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and schema
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print all record set `@id`s and the associated field `@id`s and names from the dataset. Record sets are containers for tables/records. Fields describe the columns.

In [ ]:
# List all record set @ids available in the dataset
record_sets = list(dataset.record_sets)
print("Available record sets and their fields:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  fields:")
    for field in fields:
        if isinstance(field, dict):
            # Croissant standard: field as dict
            print(f"    - @id: {field.get('@id')}, name: {field.get('name')}")
        else:
            # Croissant: field as @id string, can lookup from metadata.fields
            fobj = dataset.metadata.get_field(field)
            print(f"    - @id: {field}, name: {getattr(fobj,'name', 'N/A')}")
    print()

#### Print Sample Records
Let's print a sample of the first record from each record set. (Reference record sets by their `@id`.)

In [ ]:
for rs in record_sets:
    record_set_id = rs['@id']
    print(f"Sample record from record set {record_set_id}:")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        for i, rec in enumerate(records_iter):
            pprint(rec)
            break
    except Exception as e:
        print(f"  Could not load records: {e}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract **all record sets** and store them in a dictionary by `@id`.

In [ ]:
dataframes = dict()
print('Extracting records for each record set...')
for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded data from {rs_id}: shape = {df.shape}")
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# List columns of the largest record set (the main data table)
if dataframes:
    # Heuristic: pick record set with the most columns/rows
    main_rs_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"Most populous record set is: {main_rs_id}")
    print("Columns:", dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No data extracted from record sets.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. You can remove outliers, transform distributions, or group by attributes to prepare the data for further analysis.

Below, select a numeric field (referenced by its `@id`) and group field from the main record set.

In [ ]:
from numpy import number

# Use the main record set
record_set_id = main_rs_id
df = dataframes[record_set_id]

# Try to auto-detect a numeric field from the columns
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    print("No numeric field detected in the main data table.")
else:
    print(f"Selected numeric field: {numeric_field}")
    # Filter records by a threshold
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a categorical/group field
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing means):")
        print(grouped_df.head())
    else:
        print("No suitable group field detected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show the distribution of the selected numeric field, and if possible, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(data=df, x=numeric_field, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    # Boxplot by group field, if detected
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to explore a Croissant-based FAIR dataset. We loaded its schema, enumerated record sets and fields by their `@id`, and performed basic exploratory analysis and visualization.

You may further extend this notebook to clean, transform, or model the data according to your scientific or machine learning needs.
